# Project Phase II: Applied Data Engineering and Machine Learning

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the original dataset
df = pd.read_csv(
    "../data/student-course-completion-prediction-dataset-original.csv"
)

# Select a reproducible 5,000-row stratified sample.
# This keeps the Completed / Not Completed proportions similar
# while reducing the computational cost of eLCS.
df_copy, _ = train_test_split(
    df,
    train_size=5000,
    random_state=42,
    stratify=df["Completed"]
)

# Reset the row index after sampling
df_copy = df_copy.reset_index(drop=True)

print(f"Total selected: {len(df_copy)}")
print(df_copy["Completed"].value_counts())

Total selected: 5000
Completed
Not Completed    2549
Completed        2451
Name: count, dtype: int64


In [2]:
for col in df_copy.columns:
    print(f"{col}: {df_copy[col].dtype}, unique: {df_copy[col].nunique(dropna=True)}")

Student_ID: str, unique: 5000
Name: str, unique: 300
Gender: str, unique: 3
Age: int64, unique: 29
Education_Level: str, unique: 5
Employment_Status: str, unique: 4
City: str, unique: 15
Device_Type: str, unique: 3
Internet_Connection_Quality: str, unique: 3
Course_ID: str, unique: 8
Course_Name: str, unique: 8
Category: str, unique: 5
Course_Level: str, unique: 3
Course_Duration_Days: int64, unique: 8
Instructor_Rating: float64, unique: 7
Login_Frequency: int64, unique: 15
Average_Session_Duration_Min: int64, unique: 67
Video_Completion_Rate: float64, unique: 829
Discussion_Participation: int64, unique: 11
Time_Spent_Hours: float64, unique: 174
Days_Since_Last_Login: int64, unique: 50
Notifications_Checked: int64, unique: 17
Peer_Interaction_Score: float64, unique: 98
Assignments_Submitted: int64, unique: 11
Assignments_Missed: int64, unique: 11
Quiz_Attempts: int64, unique: 14
Quiz_Score_Avg: float64, unique: 582
Project_Grade: float64, unique: 692
Progress_Percentage: float64, uniqu

### Data Preprocessing and Feature Engineering

In [3]:
# Check and correct incorrect data types
print("=== Normalizing Data Types ===\n")

integer_columns = [
    "Age",
    "Course_Duration_Days",
    "Login_Frequency",
    "Average_Session_Duration_Min",
    "Discussion_Participation",
    "Days_Since_Last_Login",
    "Notifications_Checked",
    "Assignments_Submitted",
    "Assignments_Missed",
    "Quiz_Attempts",
    "Rewatch_Count",
    "Payment_Amount",
    "App_Usage_Percentage",
    "Reminder_Emails_Clicked",
    "Support_Tickets_Raised",
]
float_columns = [
    "Instructor_Rating",
    "Video_Completion_Rate",
    "Time_Spent_Hours",
    "Peer_Interaction_Score",
    "Quiz_Score_Avg",
    "Project_Grade",
    "Progress_Percentage",
    "Satisfaction_Rating",
]
expected_numeric_dtypes = {
    **dict.fromkeys(integer_columns, "int64"),
    **dict.fromkeys(float_columns, "float64"),
}

for column_name, expected_dtype in expected_numeric_dtypes.items():
    if column_name not in df_copy.columns:
        continue

    original_values = df_copy[column_name]
    converted_values = pd.to_numeric(original_values, errors="coerce")
    invalid_mask = original_values.notna() & converted_values.isna()

    if invalid_mask.any():
        examples = original_values[invalid_mask].head(5).tolist()
        print(
            f"{column_name}: {invalid_mask.sum()} values could not be converted "
            f"to {expected_dtype}; examples: {examples}"
        )

    fractional_mask = converted_values.notna() & converted_values.mod(1).ne(0)
    if expected_dtype == "int64" and fractional_mask.any():
        print(
            f"{column_name}: {fractional_mask.sum()} fractional values found; "
            "preserving the column as float64"
        )
        df_copy[column_name] = converted_values.astype("float64")
    elif expected_dtype == "int64" and not converted_values.isna().any():
        df_copy[column_name] = converted_values.astype("int64")
    else:
        df_copy[column_name] = converted_values.astype("float64")

# Dataset dates use day-month-year format.
if "Enrollment_Date" in df_copy.columns:
    original_dates = df_copy["Enrollment_Date"]
    parsed_dates = pd.to_datetime(original_dates, format="%d-%m-%Y", errors="coerce")
    invalid_dates = original_dates.notna() & parsed_dates.isna()

    if invalid_dates.any():
        examples = original_dates[invalid_dates].head(5).tolist()
        print(
            f"Enrollment_Date: {invalid_dates.sum()} values could not be parsed; "
            f"examples: {examples}"
        )

    df_copy["Enrollment_Date"] = parsed_dates

print("\nResulting data types:")
print(df_copy.dtypes.to_string())

=== Normalizing Data Types ===


Resulting data types:
Student_ID                                 str
Name                                       str
Gender                                     str
Age                                      int64
Education_Level                            str
Employment_Status                          str
City                                       str
Device_Type                                str
Internet_Connection_Quality                str
Course_ID                                  str
Course_Name                                str
Category                                   str
Course_Level                               str
Course_Duration_Days                     int64
Instructor_Rating                      float64
Login_Frequency                          int64
Average_Session_Duration_Min             int64
Video_Completion_Rate                  float64
Discussion_Participation                 int64
Time_Spent_Hours                       float64
Days_

In [4]:
# Address missing values and duplicate records

# Display duplicate Student_ID records
duplicate_student_rows = df_copy[df_copy["Student_ID"].duplicated(keep=False)]

print("Duplicate records with the same Student_ID:")
if len(duplicate_student_rows) > 0:
    print(duplicate_student_rows.to_string(index=False))
else:
    print("No duplicate Student_ID records found.")

# Display completely identical duplicate records
complete_duplicate_rows = df_copy[df_copy.duplicated(keep=False)]

print("\nComplete duplicate records:")
if len(complete_duplicate_rows) > 0:
    print(complete_duplicate_rows.to_string(index=False))
else:
    print("No complete duplicate records found.")

# Check columns with missing values
missing = df_copy.isna().sum()
missing_columns = missing[missing > 0].sort_values(ascending=False)

print("\nColumns containing missing values:")

if missing_columns.empty:
    print("No missing values found.")
else:
    for column_name, missing_count in missing_columns.items():
        print(f"{column_name}: {missing_count}")

    # Display rows containing missing values
    rows_with_missing = df_copy[df_copy.isna().any(axis=1)]

    print("\nRows containing missing values:")
    print(rows_with_missing.to_string(index=False))

print("\nDataset shape after inspection:")
print(f"Rows: {df_copy.shape[0]}, Columns: {df_copy.shape[1]}")

Duplicate records with the same Student_ID:
No duplicate Student_ID records found.

Complete duplicate records:
No complete duplicate records found.

Columns containing missing values:
No missing values found.

Dataset shape after inspection:
Rows: 5000, Columns: 40


In [5]:
# Inspect and fix invalid values

print("=== Inspecting Invalid Values ===\n")

# Check for negative values in columns that should be non-negative
numeric_cols = df_copy.select_dtypes(include="number").columns

invalid_values_found = False

for col in numeric_cols:
    negative_count = (df_copy[col] < 0).sum()

    if negative_count > 0:
        invalid_values_found = True
        print(f"{col}: {negative_count} negative values found")

        # Show examples of negative values
        negative_examples = df_copy[df_copy[col] < 0][col].head(5).tolist()
        print(f"  Examples: {negative_examples}")

if not invalid_values_found:
    print("No invalid numeric values detected.\n")
else:
    print()

# Check for unexpected values in expected binary/categorical columns
binary_cols = {
    "Fee_Paid": ["Yes", "No"],
    "Discount_Used": ["Yes", "No"],
    "Completed": ["Completed", "Not Completed"],
}

print("=== Inspecting Expected Categorical Values ===\n")

categorical_anomalies = False

for col, expected_values in binary_cols.items():
    if col in df_copy.columns:
        actual_values = df_copy[col].unique()
        unexpected = [
            value
            for value in actual_values
            if value not in expected_values and pd.notna(value)
        ]

        if unexpected:
            categorical_anomalies = True
            print(f"{col}: Unexpected values found")
            print(f"  Expected: {expected_values}")
            print(f"  Found: {list(set(unexpected))}")

if not categorical_anomalies:
    print("All categorical columns have expected values.\n")
else:
    print()

# Check for inconsistent categories with case/whitespace variations
print("=== Inspecting Inconsistent Categories ===\n")

category_check_cols = [
    "Gender",
    "Education_Level",
    "Employment_Status",
    "City",
    "Device_Type",
    "Internet_Connection_Quality",
    "Course_ID",
    "Course_Name",
    "Category",
    "Course_Level",
    "Payment_Mode",
    "Fee_Paid",
    "Discount_Used",
    "Completed",
]

inconsistency_found = False

for col in category_check_cols:
    if col in df_copy.columns:
        original_count = df_copy[col].nunique(dropna=False)

        cleaned_count = (
            df_copy[col].astype("string").str.strip().str.lower().nunique(dropna=False)
        )

        if original_count != cleaned_count:
            inconsistency_found = True
            print(f"{col}: {original_count} original vs {cleaned_count} standardised")

            # Show examples of variations
            original_values = df_copy[col].dropna().unique()[:10]
            print(f"  Sample values: {list(original_values)}")

if not inconsistency_found:
    print("No categorical inconsistencies detected.\n")
else:
    print()

=== Inspecting Invalid Values ===

No invalid numeric values detected.

=== Inspecting Expected Categorical Values ===

All categorical columns have expected values.

=== Inspecting Inconsistent Categories ===

No categorical inconsistencies detected.



In [ ]:
# Split before any data-dependent preprocessing to avoid data leakage.
from sklearn.model_selection import train_test_split

if not df_copy["Completed"].isin(["Not Completed", "Completed"]).all():
    raise ValueError("Completed contains labels outside the expected categories.")

# Keep 20% as the final untouched test set.
X_train_full_raw, X_test_original, y_train_full, y_test = train_test_split(
    df_copy.drop(columns=["Completed"]),
    df_copy["Completed"].map({"Not Completed": 0, "Completed": 1}),
    test_size=0.2,
    random_state=42,
    stratify=df_copy["Completed"],
)

# Split the training portion again BEFORE any fitted preprocessing.
# This validation set is used only for hyperparameter tuning.
X_train_tune_original, X_val_original, y_train_tune_original, y_val = train_test_split(
    X_train_full_raw,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full,
)

# Explicit copies protect the original splits from in-place modification.
X_train_full_raw = X_train_full_raw.copy()
X_test_original = X_test_original.copy()
X_train_tune_original = X_train_tune_original.copy()
X_val_original = X_val_original.copy()

print("=== Leakage-Safe Train / Validation / Test Split ===")
print(f"Full training rows: {len(X_train_full_raw)}")
print(f"Training-for-tuning rows: {len(X_train_tune_original)}")
print(f"Validation rows: {len(X_val_original)}")
print(f"Final test rows: {len(X_test_original)}")

print("\nTarget proportions:")
print("Training-for-tuning:")
print(y_train_tune_original.value_counts(normalize=True).sort_index().round(3))

print("\nValidation:")
print(y_val.value_counts(normalize=True).sort_index().round(3))

print("\nFinal test:")
print(y_test.value_counts(normalize=True).sort_index().round(3))


In [ ]:
# ============================================================
# Leakage-safe preprocessing for eLCS
# ============================================================

import numpy as np
from functools import partial
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, KBinsDiscretizer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.utils import resample, shuffle


def preprocess_elcs_data(
    X_fit_raw,
    y_fit_raw,
    X_apply_raw,
    branch_name="Preprocessing",
):
    """
    Fit every data-dependent preprocessing step on X_fit_raw only,
    then apply the fitted transformations to X_apply_raw.

    This function is used twice:
      1. tuning branch: fit on training-for-tuning, apply to validation
      2. final branch: fit on full training data, apply to final test data
    """

    print(f"\n{'=' * 72}")
    print(branch_name)
    print("=" * 72)

    X_fit = X_fit_raw.copy()
    X_apply = X_apply_raw.copy()
    y_fit = y_fit_raw.copy()

    initial_fit_rows = len(X_fit)

    # --------------------------------------------------------
    # 1. Investigate outliers using fitting data only
    # --------------------------------------------------------
    numeric_cols = X_fit.select_dtypes(include="number").columns

    valid_ranges = {
        "Age": (17, 80),
        "Instructor_Rating": (0, 5),
        "Satisfaction_Rating": (0, 5),
        "Video_Completion_Rate": (0, 100),
        "Progress_Percentage": (0, 100),
        "App_Usage_Percentage": (0, 100),
        "Quiz_Score_Avg": (0, 100),
        "Project_Grade": (0, 100),
        "Course_Duration_Days": (0, 90),
        "Average_Session_Duration_Min": (0, 80),
        "Time_Spent_Hours": (0, 30),
        "Days_Since_Last_Login": (0, 100),
        "Payment_Amount": (0, None),
    }

    outlier_summary = {}

    for col in numeric_cols:
        training_values = X_fit[col]
        q1 = training_values.quantile(0.25)
        q3 = training_values.quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outlier_mask = (
            (training_values < lower_bound)
            | (training_values > upper_bound)
        )

        range_violations = 0
        if col in valid_ranges:
            min_valid, max_valid = valid_ranges[col]
            if max_valid is None:
                range_mask = training_values < min_valid
            else:
                range_mask = (
                    (training_values < min_valid)
                    | (training_values > max_valid)
                )
            range_violations = int(range_mask.sum())

        outlier_summary[col] = {
            "outlier_count_iqr": int(outlier_mask.sum()),
            "outlier_pct": round(float(outlier_mask.mean() * 100), 2),
            "range_violations": range_violations,
            "min": training_values.min(),
            "max": training_values.max(),
            "iqr_bounds": (lower_bound, upper_bound),
        }

    print("Columns with IQR outliers above 0.5% or range violations:")
    displayed_any = False
    for col, stats in outlier_summary.items():
        if stats["outlier_pct"] > 0.5 or stats["range_violations"] > 0:
            displayed_any = True
            print(
                f"  {col}: IQR outliers={stats['outlier_count_iqr']} "
                f"({stats['outlier_pct']:.2f}%), "
                f"range violations={stats['range_violations']}"
            )

    if not displayed_any:
        print("  None")

    # --------------------------------------------------------
    # 2. Apply fixed domain limits
    # --------------------------------------------------------
    percentage_cols = [
        "Video_Completion_Rate",
        "Progress_Percentage",
        "App_Usage_Percentage",
        "Quiz_Score_Avg",
        "Project_Grade",
    ]

    for col in percentage_cols:
        if col in X_fit.columns:
            X_fit[col] = X_fit[col].clip(0, 100)
            X_apply[col] = X_apply[col].clip(0, 100)

    rating_cols = ["Instructor_Rating", "Satisfaction_Rating"]

    for col in rating_cols:
        if col in X_fit.columns:
            X_fit[col] = X_fit[col].clip(0, 5)
            X_apply[col] = X_apply[col].clip(0, 5)

    if "Age" in X_fit.columns:
        X_fit["Age"] = X_fit["Age"].clip(17, 80)
        X_apply["Age"] = X_apply["Age"].clip(17, 80)

    # --------------------------------------------------------
    # 3. Remove only extreme IQR outliers from fitting data
    #    Validation/test rows are never removed.
    # --------------------------------------------------------
    rows_to_remove = set()

    for col in numeric_cols:
        training_values = X_fit[col]
        q1 = training_values.quantile(0.25)
        q3 = training_values.quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 3 * iqr
        upper_bound = q3 + 3 * iqr

        extreme_mask = (
            (training_values < lower_bound)
            | (training_values > upper_bound)
        )

        if extreme_mask.any():
            rows_to_remove.update(
                X_fit.index[extreme_mask].tolist()
            )

    if rows_to_remove:
        rows_to_remove = sorted(rows_to_remove)
        X_fit = X_fit.drop(index=rows_to_remove)
        y_fit = y_fit.drop(index=rows_to_remove)

    print(
        f"Extreme fitting rows removed: "
        f"{initial_fit_rows - len(X_fit)}"
    )
    print(f"Fitting rows retained: {len(X_fit)}")
    print(f"Evaluation rows retained unchanged: {len(X_apply)}")

    # --------------------------------------------------------
    # 4. Remove identifiers / unsuitable modelling columns
    # --------------------------------------------------------
    identifier_cols = [
        "Student_ID",
        "Name",
        "Course_Name",
        "Enrollment_Date",
    ]

    X_fit = X_fit.drop(
        columns=identifier_cols,
        errors="ignore",
    )
    X_apply = X_apply.drop(
        columns=identifier_cols,
        errors="ignore",
    )

    # --------------------------------------------------------
    # 5. Encode categorical features
    # --------------------------------------------------------
    categorical_features = [
        "Gender",
        "Education_Level",
        "Employment_Status",
        "City",
        "Device_Type",
        "Internet_Connection_Quality",
        "Course_ID",
        "Category",
        "Payment_Mode",
        "Fee_Paid",
        "Discount_Used",
    ]

    categorical_features = [
        col for col in categorical_features
        if col in X_fit.columns
    ]

    for col in categorical_features:
        X_fit[col] = (
            X_fit[col]
            .astype("string")
            .fillna("__MISSING__")
        )
        X_apply[col] = (
            X_apply[col]
            .astype("string")
            .fillna("__MISSING__")
        )

    categorical_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )

    if categorical_features:
        X_fit[categorical_features] = categorical_encoder.fit_transform(
            X_fit[categorical_features]
        )
        X_apply[categorical_features] = categorical_encoder.transform(
            X_apply[categorical_features]
        )

    # Preserve the known course-level order.
    course_level_order = {
        "Beginner": 0,
        "Intermediate": 1,
        "Advanced": 2,
    }

    if "Course_Level" in X_fit.columns:
        X_fit["Course_Level"] = (
            X_fit["Course_Level"]
            .map(course_level_order)
            .fillna(-1)
            .astype("int64")
        )
        X_apply["Course_Level"] = (
            X_apply["Course_Level"]
            .map(course_level_order)
            .fillna(-1)
            .astype("int64")
        )

    if X_fit.columns.tolist() != X_apply.columns.tolist():
        raise ValueError(
            "Fitting and evaluation feature columns do not match."
        )

    # --------------------------------------------------------
    # 6. Define features that will be discretised
    # --------------------------------------------------------
    discretise_features = [
        "Age",
        "Course_Duration_Days",
        "Average_Session_Duration_Min",
        "Video_Completion_Rate",
        "Quiz_Score_Avg",
        "Project_Grade",
        "Progress_Percentage",
    ]

    discretise_features = [
        col for col in discretise_features
        if col in X_fit.columns
    ]

    # --------------------------------------------------------
    # 7. Scale continuous features that are NOT discretised
    # --------------------------------------------------------
    continuous_features = [
        "Age",
        "Course_Duration_Days",
        "Instructor_Rating",
        "Login_Frequency",
        "Average_Session_Duration_Min",
        "Video_Completion_Rate",
        "Discussion_Participation",
        "Time_Spent_Hours",
        "Days_Since_Last_Login",
        "Notifications_Checked",
        "Peer_Interaction_Score",
        "Assignments_Submitted",
        "Assignments_Missed",
        "Quiz_Attempts",
        "Quiz_Score_Avg",
        "Project_Grade",
        "Progress_Percentage",
        "Rewatch_Count",
        "Payment_Amount",
        "App_Usage_Percentage",
        "Reminder_Emails_Clicked",
        "Support_Tickets_Raised",
        "Satisfaction_Rating",
    ]

    continuous_features = [
        col
        for col in continuous_features
        if col in X_fit.columns
        and col not in discretise_features
    ]

    scaler = StandardScaler()

    if continuous_features:
        X_fit[continuous_features] = scaler.fit_transform(
            X_fit[continuous_features]
        )
        X_apply[continuous_features] = scaler.transform(
            X_apply[continuous_features]
        )

    print(f"Scaled continuous features: {len(continuous_features)}")

    # --------------------------------------------------------
    # 8. Quantile discretisation
    # --------------------------------------------------------
    discretiser = KBinsDiscretizer(
        n_bins=5,
        encode="ordinal",
        strategy="quantile",
    )

    if discretise_features:
        X_fit[discretise_features] = discretiser.fit_transform(
            X_fit[discretise_features]
        )
        X_apply[discretise_features] = discretiser.transform(
            X_apply[discretise_features]
        )

    print(f"Discretised features: {len(discretise_features)}")

    # --------------------------------------------------------
    # 9. Assess class balance using fitting labels only
    # --------------------------------------------------------
    class_counts = y_fit.value_counts().sort_index()

    if len(class_counts) < 2:
        raise ValueError(
            "Fitting data must contain both classes."
        )

    imbalance_ratio = (
        class_counts.max()
        / class_counts.min()
    )
    imbalance_threshold = 1.5

    print(
        f"Fitting majority/minority ratio: "
        f"{imbalance_ratio:.3f}:1"
    )

    # Only oversample if imbalance is substantial.
    if imbalance_ratio >= imbalance_threshold:
        majority_class = class_counts.idxmax()
        minority_class = class_counts.idxmin()

        majority_indices = y_fit[
            y_fit == majority_class
        ].index.to_numpy()

        minority_indices = y_fit[
            y_fit == minority_class
        ].index.to_numpy()

        sampled_minority_indices = resample(
            minority_indices,
            replace=True,
            n_samples=len(majority_indices),
            random_state=42,
        )

        combined_indices = np.concatenate(
            [majority_indices, sampled_minority_indices]
        )

        X_fit = X_fit.loc[combined_indices].reset_index(drop=True)
        y_fit = y_fit.loc[combined_indices].reset_index(drop=True)

        X_fit, y_fit = shuffle(
            X_fit,
            y_fit,
            random_state=42,
        )
        X_fit = X_fit.reset_index(drop=True)
        y_fit = y_fit.reset_index(drop=True)

        print("Minority class oversampled in fitting data.")
    else:
        print(
            "Classes sufficiently balanced; "
            "no resampling applied."
        )

    # --------------------------------------------------------
    # 10. Feature selection using fitting labels only
    # --------------------------------------------------------
    discrete_feature_mask = [
        (
            column_name in categorical_features
            or column_name == "Course_Level"
            or column_name in discretise_features
        )
        for column_name in X_fit.columns
    ]

    mutual_info_score = partial(
        mutual_info_classif,
        discrete_features=discrete_feature_mask,
        random_state=42,
    )

    n_selected_features = min(
        20,
        X_fit.shape[1],
    )

    feature_selector = SelectKBest(
        score_func=mutual_info_score,
        k=n_selected_features,
    )

    feature_selector.fit(
        X_fit,
        y_fit,
    )

    selected_feature_names = X_fit.columns[
        feature_selector.get_support()
    ].tolist()

    selected_feature_scores = pd.DataFrame(
        {
            "Feature": X_fit.columns,
            "Mutual Information": feature_selector.scores_,
        }
    ).sort_values(
        "Mutual Information",
        ascending=False,
    )

    X_fit_selected = pd.DataFrame(
        feature_selector.transform(X_fit),
        columns=selected_feature_names,
    )

    X_apply_selected = pd.DataFrame(
        feature_selector.transform(X_apply),
        columns=selected_feature_names,
        index=X_apply.index,
    )

    print(
        f"Selected {len(selected_feature_names)} "
        f"of {X_fit.shape[1]} features."
    )
    print(
        f"Final fitting shape: "
        f"{X_fit_selected.shape}"
    )
    print(
        f"Final evaluation shape: "
        f"{X_apply_selected.shape}"
    )
    print("Selected feature names:")
    print(selected_feature_names)

    metadata = {
        "categorical_features": categorical_features,
        "continuous_features": continuous_features,
        "discretise_features": discretise_features,
        "selected_feature_names": selected_feature_names,
        "selected_feature_scores": selected_feature_scores,
        "encoder": categorical_encoder,
        "scaler": scaler,
        "discretiser": discretiser,
        "feature_selector": feature_selector,
    }

    return (
        X_fit_selected,
        y_fit,
        X_apply_selected,
        metadata,
    )


# ============================================================
# TUNING BRANCH
# Fit preprocessing only on training-for-tuning rows.
# Validation rows are transformed but never used for fitting.
# ============================================================

(
    X_train_tune,
    y_train_tune,
    X_val,
    tuning_preprocessing,
) = preprocess_elcs_data(
    X_train_tune_original,
    y_train_tune_original,
    X_val_original,
    branch_name="TUNING PREPROCESSING",
)

print("\n=== Tuning Dataset Ready ===")
print(f"Training-for-tuning: {X_train_tune.shape}")
print(f"Validation: {X_val.shape}")


# ============================================================
# FINAL BRANCH
# Refit preprocessing on the complete training portion.
# The final test set is transformed only after fitting.
# ============================================================

(
    X_train_raw,
    y_train,
    X_test_raw,
    final_preprocessing,
) = preprocess_elcs_data(
    X_train_full_raw,
    y_train_full,
    X_test_original,
    branch_name="FINAL TRAIN/TEST PREPROCESSING",
)

# Keep these names available for later notebook cells.
categorical_features = final_preprocessing["categorical_features"]
continuous_features = final_preprocessing["continuous_features"]
discretise_features = final_preprocessing["discretise_features"]
selected_feature_names = final_preprocessing["selected_feature_names"]
selected_feature_scores = final_preprocessing["selected_feature_scores"]
categorical_encoder = final_preprocessing["encoder"]
scaler = final_preprocessing["scaler"]
discretiser = final_preprocessing["discretiser"]
feature_selector = final_preprocessing["feature_selector"]

print("\n=== Final Modeling Dataset Ready ===")
print(f"Full training data: {X_train_raw.shape}")
print(f"Untouched final test data: {X_test_raw.shape}")
print(
    "Validation preprocessing was fitted only on "
    "training-for-tuning data."
)
print(
    "Final test preprocessing was fitted only on "
    "the full training data."
)
print(
    "Final test labels were not used for fitting, "
    "feature selection, or hyperparameter tuning."
)


In [ ]:
# # PCA candidate branch using scaled continuous features only.
# # Replaces the canonical train/test feature frames in place.

# from sklearn.decomposition import PCA

# pca_features = [
#     column_name
#     for column_name in continuous_features
#     if column_name in X_train_raw.columns
# ]
# non_pca_features = [
#     column_name
#     for column_name in X_train_raw.columns
#     if column_name not in pca_features
# ]

# if not pca_features:
#     raise ValueError("No continuous features are available for PCA.")

# pca_model = PCA(n_components=0.95, svd_solver="full")
# train_components = pca_model.fit_transform(X_train_raw[pca_features])
# test_components = pca_model.transform(X_test_raw[pca_features])

# component_names = [
#     f"PC{component_number + 1}" for component_number in range(pca_model.n_components_)
# ]
# train_pca_df = pd.DataFrame(
#     train_components,
#     columns=component_names,
#     index=X_train_raw.index,
# )
# test_pca_df = pd.DataFrame(
#     test_components,
#     columns=component_names,
#     index=X_test_raw.index,
# )

# X_train_raw = pd.concat([X_train_raw[non_pca_features], train_pca_df], axis=1)
# X_test_raw = pd.concat([X_test_raw[non_pca_features], test_pca_df], axis=1)

# print(f"PCA components retained: {pca_model.n_components_}")
# print(
#     "Continuous-feature variance retained: "
#     f"{pca_model.explained_variance_ratio_.sum():.3f}"
# )
# print(f"PCA training shape: {X_train_raw.shape}")
# print(f"PCA test shape: {X_test_raw.shape}")
# print("Features after PCA:")
# print(X_train_raw.columns.tolist())
# print(
#     "PCA was fitted on training data only; encoded categorical features were retained."
# )

### Improved LCS-Based System

#### eLCS Hyperparameter Tuning


In [ ]:
import matplotlib.pyplot as plt
from skeLCS import eLCS
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
)


def compute_elcs_metrics(label, y_test, y_pred, probabilities):
    """Compute and return standard evaluation metrics, ROC, PRC points, and confusion matrix."""
    false_negative_cost = 10.0
    false_positive_cost = 1.0
    # False Negative:
    # Actual = Completed (1), Predicted = Not Completed (0)
    false_negatives = ((y_test == 1) & (y_pred == 0)).sum()

    # False Positive:
    # Actual = Not Completed (0), Predicted = Completed (1)
    false_positives = ((y_test == 0) & (y_pred == 1)).sum()
    mean_cost = (
        false_negative_cost * false_negatives + false_positive_cost * false_positives
    ) / len(y_test)

    fpr, tpr, _ = roc_curve(y_test, probabilities[:, 1])
    precision_curve, recall_curve, _ = precision_recall_curve(
        y_test, probabilities[:, 1]
    )

    cm = confusion_matrix(y_test, y_pred)

    metrics = {
        "label": label,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "mean_cost": mean_cost,
        "roc_auc": auc(fpr, tpr),
        "prc_auc": auc(recall_curve, precision_curve),
    }
    return metrics, (fpr, tpr), (recall_curve, precision_curve), cm


def run_elcs_experiment(params, X_train, y_train, X_test, y_test, label=None):
    """Train one eLCS model and retain test predictions for paired comparisons."""
    label = label or str(params)
    print(f"Executing configuration '{label}' with parameters: {params}")

    train_X = X_train.values if hasattr(X_train, "values") else X_train
    train_y = y_train.values if hasattr(y_train, "values") else y_train
    test_X = X_test.values if hasattr(X_test, "values") else X_test

    model = eLCS(**params)
    model.fit(train_X, train_y)

    y_pred = model.predict(test_X)
    probabilities = model.predict_proba(test_X)

    metrics, roc_data, prc_data, cm = compute_elcs_metrics(
        label, y_test, y_pred, probabilities
    )

    return metrics, model, roc_data, prc_data, cm, np.asarray(y_pred)


def hypertune_elcs(param_grid, X_train, y_train, X_test, y_test):
    """Run eLCS configurations and retain per-row predictions for statistical testing."""
    results = []
    roc_curves = {}
    prc_curves = {}
    confusion_matrices = {}
    predictions = {}

    for label, params in param_grid.items():
        metrics, _, roc_data, prc_data, cm, y_pred = run_elcs_experiment(
            params, X_train, y_train, X_test, y_test, label=label
        )
        results.append(metrics)
        roc_curves[label] = roc_data
        prc_curves[label] = prc_data
        confusion_matrices[label] = cm
        predictions[label] = y_pred

    results_df = pd.DataFrame(results)
    results_df.attrs["roc_curves"] = roc_curves
    results_df.attrs["prc_curves"] = prc_curves
    results_df.attrs["confusion_matrices"] = confusion_matrices
    results_df.attrs["predictions"] = predictions

    return results_df

In [ ]:
# Run the baseline and randomized eLCS configurations.
import random

random.seed(42)

param_ranges = {
    "learning_iterations": {"type": "int", "min": 500, "max": 15000, "step": 100},
    "N": {"type": "int", "min": 300, "max": 1500, "step": 100},
    "p_spec": {"type": "float", "min": 0.1, "max": 0.8, "digits": 2},
    "nu": {"type": "float", "min": 1.0, "max": 10.0, "digits": 1},
    "chi": {"type": "float", "min": 0.4, "max": 0.95, "digits": 2},
    "mu": {"type": "float", "min": 0.01, "max": 0.15, "digits": 3},
    "theta_sel": {"type": "float", "min": 0.1, "max": 0.8, "digits": 2},
    "acc_sub": {"type": "float", "min": 0.90, "max": 1.0, "digits": 2},
    "selection_method": {"type": "choice", "values": ["tournament", "roulette"]},
    "do_correct_set_subsumption": {"type": "choice", "values": [False, True]},
}


def sample_parameter_value(spec):
    """Sample a value from an integer, float, or categorical parameter range."""
    param_type = spec["type"]
    if param_type == "int":
        step = spec.get("step", 1)
        return random.randrange(spec["min"], spec["max"] + step, step)
    if param_type == "float":
        digits = spec.get("digits", 2)
        return round(random.uniform(spec["min"], spec["max"]), digits)
    if param_type == "choice":
        return random.choice(spec["values"])
    raise ValueError(f"Unknown parameter type: {param_type}")


n_random_configs = 10
elcs_param_grid = {"baseline": dict()}

for config_number in range(1, n_random_configs + 1):
    config = {
        param_name: sample_parameter_value(spec)
        for param_name, spec in param_ranges.items()
    }
    config["track_accuracy_while_fit"] = False
    config["random_state"] = 42 + config_number
    elcs_param_grid[f"random_config_{config_number}"] = config

# Tune configurations using validation data, not the final test set
elcs_hypertune_results = hypertune_elcs(
    elcs_param_grid,
    X_train_tune,
    y_train_tune,
    X_val,
    y_val,
)

In [ ]:
# Pairwise statistical comparison (paired t-test) of baseline and randomized eLCS configurations.
# Balanced accuracy is the comparison metric.
from itertools import combinations
from scipy.stats import ttest_rel
from sklearn.metrics import balanced_accuracy_score

alpha = 0.05
bootstrap_count = 1000
random_generator = np.random.default_rng(42)
# Configuration comparisons are performed on validation data
y_test_values = np.asarray(y_val)
configuration_predictions = elcs_hypertune_results.attrs.get("predictions", {})
configuration_labels = list(elcs_param_grid)

if set(configuration_labels) != set(configuration_predictions):
    raise ValueError(
        "Per-row predictions are missing or do not match the configured models. "
        "Rerun the experiment cell after updating hypertune_elcs."
    )

class_indices = {
    class_label: np.flatnonzero(y_test_values == class_label)
    for class_label in np.unique(y_test_values)
}
if len(class_indices) != 2:
    raise ValueError("Stratified bootstrap comparison requires both test classes.")

bootstrap_balanced_accuracy = {
    label: np.empty(bootstrap_count) for label in configuration_labels
}

for bootstrap_index in range(bootstrap_count):
    sample_indices = np.concatenate(
        [
            random_generator.choice(indices, size=len(indices), replace=True)
            for indices in class_indices.values()
        ]
    )
    sampled_actual = y_test_values[sample_indices]

    for label in configuration_labels:
        bootstrap_balanced_accuracy[label][bootstrap_index] = balanced_accuracy_score(
            sampled_actual,
            configuration_predictions[label][sample_indices],
        )

pairwise_rows = []
raw_p_values = []
for model_a, model_b in combinations(configuration_labels, 2):
    test_result = ttest_rel(
        bootstrap_balanced_accuracy[model_a],
        bootstrap_balanced_accuracy[model_b],
    )
    mean_difference = float(
        np.mean(
            bootstrap_balanced_accuracy[model_b] - bootstrap_balanced_accuracy[model_a]
        )
    )
    raw_p_values.append(test_result.pvalue)
    pairwise_rows.append(
        {
            "Model A": model_a,
            "Model B": model_b,
            "Mean balanced-accuracy difference (B - A)": mean_difference,
            "t-test p-value": test_result.pvalue,
        }
    )

# Holm correction accounts for testing every pair of configurations.
p_values_for_adjustment = np.nan_to_num(raw_p_values, nan=1.0)
order = np.argsort(p_values_for_adjustment)
adjusted_p_values = np.ones(len(raw_p_values))
running_adjusted_p = 0.0
for rank, comparison_index in enumerate(order):
    adjusted_p = min(
        1.0,
        (len(raw_p_values) - rank) * p_values_for_adjustment[comparison_index],
    )
    running_adjusted_p = max(running_adjusted_p, adjusted_p)
    adjusted_p_values[comparison_index] = running_adjusted_p

pairwise_wins = {label: 0 for label in configuration_labels}
for row, adjusted_p_value in zip(pairwise_rows, adjusted_p_values):
    row["Holm-adjusted p-value"] = adjusted_p_value
    row["Significant difference"] = adjusted_p_value < alpha
    if adjusted_p_value < alpha:
        if row["Mean balanced-accuracy difference (B - A)"] > 0:
            pairwise_wins[row["Model B"]] += 1
        elif row["Mean balanced-accuracy difference (B - A)"] < 0:
            pairwise_wins[row["Model A"]] += 1

pairwise_ttest_results = pd.DataFrame(pairwise_rows)
display(
    pairwise_ttest_results.style.format(
        {
            "Mean balanced-accuracy difference (B - A)": "{:+.4f}",
            "t-test p-value": "{:.4g}",
            "Holm-adjusted p-value": "{:.4g}",
        }
    ).hide(axis="index")
)

random_configuration_labels = [
    label for label in configuration_labels if label != "baseline"
]
if not random_configuration_labels:
    raise ValueError("No randomized eLCS configurations are available to select.")

observed_balanced_accuracy = {
    label: balanced_accuracy_score(y_test_values, configuration_predictions[label])
    for label in configuration_labels
}

# The baseline remains in the pairwise comparisons but is never a winner candidate.
unique_random_winners = [
    label
    for label in random_configuration_labels
    if pairwise_wins[label] == len(configuration_labels) - 1
]
if len(unique_random_winners) == 1:
    best_config_label = unique_random_winners[0]
    print(
        "A randomized configuration significantly outperformed every other configuration."
    )
else:
    maximum_pairwise_wins = max(
        pairwise_wins[label] for label in random_configuration_labels
    )
    tied_random_configurations = [
        label
        for label in random_configuration_labels
        if pairwise_wins[label] == maximum_pairwise_wins
    ]
    best_config_label = max(
        tied_random_configurations,
        key=lambda label: observed_balanced_accuracy[label],
    )
    print(
        "No randomized configuration significantly outperformed every other configuration. "
        "Reporting the randomized configuration with the most significant pairwise wins; "
        "ties are broken by observed balanced accuracy."
    )

# Reuse the metrics already calculated and stored by the experiment cell.
results_by_label = elcs_hypertune_results.set_index("label")
comparison_keys = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "balanced_accuracy",
    "mean_cost",
    "roc_auc",
    "prc_auc",
]
baseline_row = results_by_label.loc["baseline"]
best_row = results_by_label.loc[best_config_label]
baseline = {key: baseline_row[key] for key in comparison_keys}
best_accuracy = {key: best_row[key] for key in comparison_keys}
best_accuracy["label"] = f"Best-Random configuration ({best_config_label})"
best_params = elcs_param_grid[best_config_label]
stored_metrics_df = pd.DataFrame(
    [
        {"Name": "baseline", **baseline},
        {"Name": "Best-Random configuration", **best_accuracy},
    ]
)

confusion_matrices = elcs_hypertune_results.attrs.get("confusion_matrices", {})
baseline_cm = confusion_matrices.get("baseline")
best_cm = confusion_matrices.get(best_config_label)

print("\n=== BASELINE eLCS (COMPARATOR) ===")
print(f"Balanced accuracy: {baseline_row['balanced_accuracy']:.4f}")
print("Parameters: default eLCS parameters")

print(f"\n=== BEST-RANDOM eLCS CONFIGURATION: {best_config_label} ===")
print(f"Balanced accuracy: {best_row['balanced_accuracy']:.4f}")
print(
    f"Significant pairwise wins: {pairwise_wins[best_config_label]} of {len(configuration_labels) - 1}"
)
print("Parameters:")
for parameter_name, parameter_value in best_params.items():
    print(f"  {parameter_name}: {parameter_value}")

print(
    "Pairwise tests use 1,000 shared stratified bootstrap resamples of the "
    "validation set. The validation set was used to compare configurations; "
    "the final test set remained untouched until final evaluation."
)

# Plot observed balanced accuracy and 95% stratified-bootstrap intervals.
scores = [
    observed_balanced_accuracy[configuration_label]
    for configuration_label in configuration_labels
]
bootstrap_intervals = {
    configuration_label: np.percentile(
        bootstrap_balanced_accuracy[configuration_label],
        [2.5, 97.5],
    )
    for configuration_label in configuration_labels
}
lower_errors = [
    score - bootstrap_intervals[configuration_label][0]
    for configuration_label, score in zip(configuration_labels, scores)
]
upper_errors = [
    bootstrap_intervals[configuration_label][1] - score
    for configuration_label, score in zip(configuration_labels, scores)
]
bar_colors = [
    (
        "#4263eb"
        if configuration_label == "baseline"
        else "#e76f51" if configuration_label == best_config_label else "#9aa0a6"
    )
    for configuration_label in configuration_labels
]

figure, axis = plt.subplots(figsize=(12, 5))
axis.bar(
    range(len(configuration_labels)),
    scores,
    yerr=np.array([lower_errors, upper_errors]),
    capsize=4,
    color=bar_colors,
)
axis.set_xticks(
    range(len(configuration_labels)),
    labels=configuration_labels,
    rotation=35,
    ha="right",
)
axis.set_ylabel("Balanced accuracy")
axis.set_ylim(0, 1.05)
axis.set_title("eLCS Configurations: Balanced Accuracy with 95% Bootstrap Intervals")
axis.grid(axis="y", linestyle="--", alpha=0.35)
axis.set_axisbelow(True)
figure.tight_layout()
plt.show()

In [ ]:
# Final evaluation on the untouched test set.
# Both eLCS models are retrained using the full training data.

(
    baseline_test_metrics,
    baseline_test_model,
    baseline_test_roc,
    baseline_test_prc,
    baseline_test_cm,
    baseline_test_pred,
) = run_elcs_experiment(
    {},
    X_train_raw,
    y_train,
    X_test_raw,
    y_test,
    label="Baseline eLCS - Final Test",
)

(
    best_test_metrics,
    best_test_model,
    best_test_roc,
    best_test_prc,
    best_test_cm,
    best_test_pred,
) = run_elcs_experiment(
    best_params,
    X_train_raw,
    y_train,
    X_test_raw,
    y_test,
    label=f"Best-Random eLCS ({best_config_label}) - Final Test",
)

print("\n=== FINAL TEST RESULTS ===")
print(
    f"Baseline balanced accuracy: "
    f"{baseline_test_metrics['balanced_accuracy']:.4f}"
)
print(
    f"Best-Random balanced accuracy: "
    f"{best_test_metrics['balanced_accuracy']:.4f}"
)


In [ ]:
# Plot comparison graphs for Baseline vs the Best-Random eLCS configuration.
import matplotlib.patheffects as path_effects

baseline_roc = baseline_test_roc
best_roc = best_test_roc
baseline_prc = baseline_test_prc
best_prc = best_test_prc

# Plot confusion matrices side by side. Rows are actual classes; columns are predictions.
confusion_matrices_to_plot = [
    ("Baseline", baseline_test_cm),
    (f"Best-Random: {best_config_label}", best_test_cm),
]
class_labels = ["Not Completed", "Completed"]
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

for axis, (model_label, matrix) in zip(axes, confusion_matrices_to_plot):
    image = axis.imshow(matrix, cmap="Blues")
    axis.set_title(model_label)
    axis.set_xlabel("Predicted class")
    axis.set_ylabel("Actual class")
    axis.set_xticks(
        range(len(class_labels)), labels=class_labels, rotation=20, ha="right"
    )
    axis.set_yticks(range(len(class_labels)), labels=class_labels)
    axis.set_xticks([index - 0.5 for index in range(matrix.shape[1] + 1)], minor=True)
    axis.set_yticks([index - 0.5 for index in range(matrix.shape[0] + 1)], minor=True)
    axis.grid(which="minor", color="#24313D", linestyle="-", linewidth=1.5)
    axis.tick_params(which="minor", bottom=False, left=False)

    threshold = matrix.max() / 2
    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            value = matrix[row_index, column_index]
            text_color = "white" if value > threshold else "#111111"
            outline_color = "#111111" if text_color == "white" else "white"
            axis.text(
                column_index,
                row_index,
                str(value),
                ha="center",
                va="center",
                fontsize=16,
                fontweight="bold",
                color=text_color,
                path_effects=[
                    path_effects.withStroke(linewidth=2, foreground=outline_color)
                ],
            )

    fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)

fig.suptitle("Confusion Matrices: Baseline vs Best-Random eLCS")
plt.show()

plt.figure(figsize=(14, 10))
plt.plot(
    baseline_roc[0],
    baseline_roc[1],
    label=f"Baseline (AUC = {baseline['roc_auc']:.4f})",
)
plt.plot(
    best_roc[0],
    best_roc[1],
    label=f"Best-Random: {best_config_label} (AUC = {best_accuracy['roc_auc']:.4f})",
)
plt.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: Baseline vs Best-Random eLCS")
plt.grid(True, linestyle="--", alpha=0.35)
plt.legend()
plt.show()

plt.figure(figsize=(14, 10))
plt.plot(
    baseline_prc[1],
    baseline_prc[0],
    label=f"Baseline (AUC = {baseline['prc_auc']:.4f})",
)
plt.plot(
    best_prc[1],
    best_prc[0],
    label=f"Best-Random: {best_config_label} (AUC = {best_accuracy['prc_auc']:.4f})",
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve: Baseline vs Best-Random eLCS")
plt.grid(True, linestyle="--", alpha=0.35)
plt.legend()
plt.show()

elcs_metrics_to_plot = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "balanced_accuracy",
    "roc_auc",
]
metrics_axis = stored_metrics_df.set_index("Name")[elcs_metrics_to_plot].plot(
    kind="bar",
    figsize=(14, 10),
)
metrics_axis.set_axisbelow(True)
metrics_axis.grid(axis="y", linestyle="--", alpha=0.35)
metrics_axis.set_ylabel("Score")
metrics_axis.set_title("eLCS Metrics: Baseline vs Best-Random Configuration")
metrics_axis.tick_params(axis="x", rotation=0)
metrics_axis.figure.tight_layout()
plt.show()

cost_axis = stored_metrics_df.set_index("Name")["mean_cost"].plot(
    kind="bar",
    figsize=(14, 10),
    color=["steelblue", "darkorange"],
)
cost_axis.set_axisbelow(True)
cost_axis.grid(axis="y", linestyle="--", alpha=0.35)
cost_axis.set_ylabel("Mean cost (lower is better)")
cost_axis.set_title("Mean Cost: Baseline vs Best-Random Configuration")
cost_axis.tick_params(axis="x", rotation=0)
cost_axis.figure.tight_layout()
plt.show()


In [ ]:
# Extract top 3 rules from the baseline and Best-Random configurations.
# Rule strength is a heuristic: fitness multiplied by classifier numerosity.

# Train the baseline eLCS configuration and export its final rule population.
baseline_rule_model = eLCS(**elcs_param_grid["baseline"])
baseline_rule_model.fit(
    X_train_raw.to_numpy(),
    y_train.to_numpy(),
)

baseline_rule_file = "../data/export/elcs_rules_baseline.csv"
baseline_rule_model.export_final_rule_population(
    headerNames=X_train_raw.columns.to_numpy(),
    className="Completed",
    filename=baseline_rule_file,
    DCAL=False,
)

baseline_rule_population = pd.read_csv(baseline_rule_file)
baseline_rule_population["Rule Strength"] = (
    baseline_rule_population["Fitness"] * baseline_rule_population["Numerosity"]
)
baseline_rule_population.insert(0, "Configuration", "baseline")

important_elcs_baseline_rules = (
    baseline_rule_population.sort_values(
        ["Rule Strength", "Accuracy", "Match Count"],
        ascending=False,
    )
    .head(3)
    .copy()
    .reset_index(drop=True)
)
important_elcs_baseline_rules.insert(
    0, "Rank", range(1, len(important_elcs_baseline_rules) + 1)
)

print("Top 3 rules from the baseline eLCS configuration:")
display(important_elcs_baseline_rules.style.hide(axis="index"))
print(f"Exported baseline rule population to {baseline_rule_file}.")

# Train the Best-Random configuration and export its final rule population.
rule_model = eLCS(**elcs_param_grid[best_config_label])
rule_model.fit(
    X_train_raw.to_numpy(),
    y_train.to_numpy(),
)

best_rule_file = "../data/export/elcs_rules_best_random.csv"
rule_model.export_final_rule_population(
    headerNames=X_train_raw.columns.to_numpy(),
    className="Completed",
    filename=best_rule_file,
    DCAL=False,
)

rule_population = pd.read_csv(best_rule_file)
rule_population["Rule Strength"] = (
    rule_population["Fitness"] * rule_population["Numerosity"]
)
rule_population.insert(0, "Configuration", best_config_label)

important_elcs_rules = (
    rule_population.sort_values(
        ["Rule Strength", "Accuracy", "Match Count"],
        ascending=False,
    )
    .head(3)
    .copy()
    .reset_index(drop=True)
)
important_elcs_rules.insert(0, "Rank", range(1, len(important_elcs_rules) + 1))

print(f"\nTop 3 rules from the Best-Random configuration ({best_config_label}):")
display(important_elcs_rules.style.hide(axis="index"))
print(f"Exported Best-Random rule population to {best_rule_file}.")

#### Using conventional models 

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix
import matplotlib.patheffects as path_effects

conventional_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest (2000 trees)": RandomForestClassifier(
        n_estimators=2000,
        random_state=42,
        n_jobs=-1,
    ),
}

conventional_results = []
conventional_roc_curves = {}
conventional_prc_curves = {}
conventional_confusion_matrices = {}
false_negative_cost = 10.0
false_positive_cost = 1.0

for model_label, model in conventional_models.items():
    model.fit(X_train_raw, y_train)
    y_pred = model.predict(X_test_raw)
    probabilities = model.predict_proba(X_test_raw)
    positive_class_index = list(model.classes_).index(1)
    positive_probabilities = probabilities[:, positive_class_index]

    # False Negative: actually Completed (1), predicted Not Completed (0)
    false_negatives = ((y_test == 1) & (y_pred == 0)).sum()

    # False Positive: actually Not Completed (0), predicted Completed (1)
    false_positives = ((y_test == 0) & (y_pred == 1)).sum()
    mean_cost = (
        false_negative_cost * false_negatives + false_positive_cost * false_positives
    ) / len(y_test)

    fpr, tpr, _ = roc_curve(y_test, positive_probabilities)
    precision_curve, recall_curve, _ = precision_recall_curve(
        y_test, positive_probabilities
    )
    model_confusion_matrix = confusion_matrix(y_test, y_pred, labels=[0, 1])

    metrics = {
        "label": model_label,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "mean_cost": mean_cost,
        "roc_auc": auc(fpr, tpr),
        "prc_auc": auc(recall_curve, precision_curve),
    }
    conventional_results.append(metrics)
    conventional_roc_curves[model_label] = (fpr, tpr)
    conventional_prc_curves[model_label] = (recall_curve, precision_curve)
    conventional_confusion_matrices[model_label] = model_confusion_matrix

    print(f"\n{model_label}")
    print("-" * len(model_label))
    for metric_name, metric_value in metrics.items():
        if metric_name != "label":
            print(f"{metric_name:>18}: {metric_value:.4f}")
    print("Confusion Matrix:")
    print(model_confusion_matrix)

conventional_results_df = (
    pd.DataFrame(conventional_results)
    .sort_values("balanced_accuracy", ascending=False)
    .reset_index(drop=True)
)
conventional_results_df.insert(0, "Rank", range(1, len(conventional_results_df) + 1))

print("\nConventional classifiers ranked by balanced accuracy:")
display(conventional_results_df.style.hide(axis="index"))

# Use a 2x2 layout so class labels and the shared colorbar do not crowd the panels.
class_labels = ["Not Completed", "Completed"]
confusion_matrix_max = max(
    matrix.max() for matrix in conventional_confusion_matrices.values()
)
fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
axes = axes.ravel()

for axis, (model_label, matrix) in zip(axes, conventional_confusion_matrices.items()):
    image = axis.imshow(matrix, cmap="Blues", vmin=0, vmax=confusion_matrix_max)
    axis.set_title(model_label)
    axis.set_xlabel("Predicted class")
    axis.set_ylabel("Actual class")
    axis.set_xticks(
        range(len(class_labels)), labels=class_labels, rotation=20, ha="right"
    )
    axis.set_yticks(range(len(class_labels)), labels=class_labels)
    axis.set_xticks([index - 0.5 for index in range(matrix.shape[1] + 1)], minor=True)
    axis.set_yticks([index - 0.5 for index in range(matrix.shape[0] + 1)], minor=True)
    axis.grid(which="minor", color="#24313D", linestyle="-", linewidth=1.5)
    axis.tick_params(which="minor", bottom=False, left=False)

    threshold = matrix.max() / 2
    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            value = matrix[row_index, column_index]
            text_color = "white" if value > threshold else "#111111"
            outline_color = "#111111" if text_color == "white" else "white"
            axis.text(
                column_index,
                row_index,
                str(value),
                ha="center",
                va="center",
                fontsize=16,
                fontweight="bold",
                color=text_color,
                path_effects=[
                    path_effects.withStroke(linewidth=2, foreground=outline_color)
                ],
            )

axes[-1].set_visible(False)
fig.colorbar(
    image, ax=axes[: len(conventional_confusion_matrices)], shrink=0.8, pad=0.04
)
fig.suptitle("Confusion Matrices: Conventional Models")
plt.show()

plt.figure(figsize=(14, 10))
for model_label, (fpr, tpr) in conventional_roc_curves.items():
    plt.plot(fpr, tpr, label=model_label)
plt.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison Across Conventional Models")
plt.grid(True, linestyle="--", alpha=0.35)
plt.legend()
plt.show()

plt.figure(figsize=(14, 10))
for model_label, (recall_curve, precision_curve) in conventional_prc_curves.items():
    plt.plot(recall_curve, precision_curve, label=model_label)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve Comparison Across Conventional Models")
plt.grid(True, linestyle="--", alpha=0.35)
plt.legend()
plt.show()

conventional_metrics_to_plot = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "balanced_accuracy",
    "roc_auc",
]
metrics_axis = conventional_results_df.set_index("label")[
    conventional_metrics_to_plot
].plot(
    kind="bar",
    figsize=(14, 10),
)
metrics_axis.set_axisbelow(True)
metrics_axis.grid(axis="y", linestyle="--", alpha=0.35)
metrics_axis.set_ylabel("Score")
metrics_axis.set_title("Conventional Model Comparison")
metrics_axis.tick_params(axis="x", rotation=45)
metrics_axis.figure.tight_layout()
plt.show()

cost_axis = conventional_results_df.set_index("label")["mean_cost"].plot(
    kind="bar", figsize=(14, 10)
)
cost_axis.set_axisbelow(True)
cost_axis.grid(axis="y", linestyle="--", alpha=0.35)
cost_axis.set_ylabel("Mean cost (lower is better)")
cost_axis.set_title("Mean Cost Across Conventional Models")
cost_axis.tick_params(axis="x", rotation=45)
cost_axis.figure.tight_layout()
plt.show()


#### Pairwise statistical comparison (paired t-test) between Baseline, Best-Random eLCS configuration, and conventional models

In [ ]:
# Compare balanced accuracy across baseline, Best-Random eLCS, and conventional models.
from itertools import combinations
from sklearn.metrics import balanced_accuracy_score

# Use eLCS predictions from the final untouched test set
comparison_predictions = {
    "Baseline eLCS": baseline_test_pred,
    f"Best-Random eLCS ({best_config_label})": best_test_pred,
}
comparison_predictions.update(
    {
        model_label: model.predict(X_test_raw)
        for model_label, model in conventional_models.items()
    }
)

comparison_labels = list(comparison_predictions)
# Final model comparisons are performed on the untouched test data
y_test_values = np.asarray(y_test)
class_indices = {
    class_label: np.flatnonzero(y_test_values == class_label)
    for class_label in np.unique(y_test_values)
}
if len(class_indices) != 2:
    raise ValueError(
        "Stratified balanced-accuracy comparison requires both test classes."
    )

bootstrap_count = 1000
random_generator = np.random.default_rng(42)
bootstrap_scores = {label: np.empty(bootstrap_count) for label in comparison_labels}

for bootstrap_index in range(bootstrap_count):
    sample_indices = np.concatenate(
        [
            random_generator.choice(indices, size=len(indices), replace=True)
            for indices in class_indices.values()
        ]
    )
    sampled_actual = y_test_values[sample_indices]

    for label in comparison_labels:
        bootstrap_scores[label][bootstrap_index] = balanced_accuracy_score(
            sampled_actual,
            comparison_predictions[label][sample_indices],
        )

comparison_rows = []
raw_p_values = []
for model_a, model_b in combinations(comparison_labels, 2):
    observed_difference = balanced_accuracy_score(
        y_test_values, comparison_predictions[model_b]
    ) - balanced_accuracy_score(y_test_values, comparison_predictions[model_a])
    bootstrap_differences = bootstrap_scores[model_b] - bootstrap_scores[model_a]
    centered_differences = bootstrap_differences - observed_difference
    p_value = (
        1 + np.count_nonzero(np.abs(centered_differences) >= abs(observed_difference))
    ) / (bootstrap_count + 1)
    confidence_interval = np.percentile(bootstrap_differences, [2.5, 97.5])

    raw_p_values.append(p_value)
    comparison_rows.append(
        {
            "Model A": model_a,
            "Model B": model_b,
            "Balanced-accuracy difference (B - A)": observed_difference,
            "95% paired-bootstrap CI": (
                f"[{confidence_interval[0]:+.4f}, {confidence_interval[1]:+.4f}]"
            ),
            "Centered-bootstrap p-value": p_value,
        }
    )

# Holm correction controls family-wise error across all pairwise comparisons.
p_values_for_adjustment = np.nan_to_num(raw_p_values, nan=1.0)
order = np.argsort(p_values_for_adjustment)
adjusted_p_values = np.ones(len(raw_p_values))
running_adjusted_p = 0.0
for rank, comparison_index in enumerate(order):
    adjusted_p = min(
        1.0,
        (len(raw_p_values) - rank) * p_values_for_adjustment[comparison_index],
    )
    running_adjusted_p = max(running_adjusted_p, adjusted_p)
    adjusted_p_values[comparison_index] = running_adjusted_p

for row, adjusted_p_value in zip(comparison_rows, adjusted_p_values):
    row["Holm-adjusted p-value"] = adjusted_p_value
    row["Significant at 0.05"] = adjusted_p_value < 0.05

balanced_accuracy_comparison = pd.DataFrame(comparison_rows)
display(
    balanced_accuracy_comparison.style.format(
        {
            "Balanced-accuracy difference (B - A)": "{:+.4f}",
            "Centered-bootstrap p-value": "{:.4g}",
            "Holm-adjusted p-value": "{:.4g}",
        }
    ).hide(axis="index")
)

observed_scores = {
    label: balanced_accuracy_score(y_test_values, comparison_predictions[label])
    for label in comparison_labels
}
print("Observed balanced accuracy:")
for label, score in observed_scores.items():
    print(f"  {label}: {score:.4f}")
print(
    "Pairwise tests use 1,000 shared stratified bootstrap resamples with centered "
    "two-sided p-values and Holm correction. The test set was used for model "
    "comparison, so treat results as exploratory."
)

# Plot observed scores with paired-bootstrap intervals.
scores = [observed_scores[label] for label in comparison_labels]
bootstrap_intervals = {
    label: np.percentile(bootstrap_scores[label], [2.5, 97.5])
    for label in comparison_labels
}
lower_errors = [
    score - bootstrap_intervals[label][0]
    for label, score in zip(comparison_labels, scores)
]
upper_errors = [
    bootstrap_intervals[label][1] - score
    for label, score in zip(comparison_labels, scores)
]
bar_colors = [
    (
        "#4263eb"
        if label == "Baseline eLCS"
        else "#e76f51" if label.startswith("Best-Random eLCS (") else "#9aa0a6"
    )
    for label in comparison_labels
]

figure, axis = plt.subplots(figsize=(12, 6))
axis.bar(
    range(len(comparison_labels)),
    scores,
    yerr=np.array([lower_errors, upper_errors]),
    capsize=4,
    color=bar_colors,
)
axis.set_xticks(
    range(len(comparison_labels)),
    labels=comparison_labels,
    rotation=30,
    ha="right",
)
axis.set_ylabel("Balanced accuracy")
axis.set_ylim(0, 1.05)
axis.set_title("Balanced Accuracy: Baseline, Best-Random eLCS, and Conventional Models")
axis.grid(axis="y", linestyle="--", alpha=0.35)
axis.set_axisbelow(True)
figure.tight_layout()
plt.show()

# Plot confusion matrices using final untouched test-set predictions
comparison_confusion_matrices = {
    "Baseline eLCS": confusion_matrix(
        y_test,
        baseline_test_pred,
        labels=[0, 1]
    ),
    f"Best-Random eLCS ({best_config_label})": confusion_matrix(
        y_test,
        best_test_pred,
        labels=[0, 1]
    ),
}
comparison_confusion_matrices.update(conventional_confusion_matrices)

class_labels = ["Not Completed", "Completed"]
confusion_matrix_max = max(
    matrix.max() for matrix in comparison_confusion_matrices.values()
)
fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
axes = axes.ravel()

for axis, (model_label, matrix) in zip(axes, comparison_confusion_matrices.items()):
    image = axis.imshow(matrix, cmap="Blues", vmin=0, vmax=confusion_matrix_max)
    axis.set_title(model_label)
    axis.set_xlabel("Predicted class")
    axis.set_ylabel("Actual class")
    axis.set_xticks(
        range(len(class_labels)), labels=class_labels, rotation=20, ha="right"
    )
    axis.set_yticks(range(len(class_labels)), labels=class_labels)
    axis.set_xticks([index - 0.5 for index in range(matrix.shape[1] + 1)], minor=True)
    axis.set_yticks([index - 0.5 for index in range(matrix.shape[0] + 1)], minor=True)
    axis.grid(which="minor", color="#24313D", linestyle="-", linewidth=1.5)
    axis.tick_params(which="minor", bottom=False, left=False)

    threshold = matrix.max() / 2
    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            value = matrix[row_index, column_index]
            text_color = "white" if value > threshold else "#111111"
            outline_color = "#111111" if text_color == "white" else "white"
            axis.text(
                column_index,
                row_index,
                str(value),
                ha="center",
                va="center",
                fontsize=16,
                fontweight="bold",
                color=text_color,
                path_effects=[
                    path_effects.withStroke(
                        linewidth=2, foreground=outline_color
                    )
                ],
            )

axes[-1].set_visible(False)
fig.colorbar(
    image, ax=axes[: len(comparison_confusion_matrices)], shrink=0.8, pad=0.04
)
fig.suptitle("Confusion Matrices: eLCS and Conventional Models")
plt.show()
